# 02 — LRGB Peptides-func: real-data evidence

The synthetic bottleneck (notebooks 04/05) is a worst case for over-squashing. Here we test Walk Attention on
a **real** long-range benchmark: **Peptides-func** from the Long Range Graph Benchmark (Dwivedi et al., 2022)
— 10,873 / 2,331 / 2,331 molecular graphs, median 138 nodes, multi-label graph classification, metric AP.

**Honest headline:** Walk Attention is *competitive* with strong baselines here (beats GCN, ties GAT) but does
**not** surpass them — Peptides has parallel paths but only *mild* over-squashing, so multi-hop reach is not
decisive. Its advantage is regime-dependent (see the RingTransfer experiment, notebook 06, for the severe
regime where it wins outright).

## 1. Load Peptides-func and inspect it

Atom features are integer category indices (models start with an embedding); the task is graph-level
multi-label (models end with mean-pooling).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from torch_geometric.datasets import LRGBDataset
tr = LRGBDataset(root='data/lrgb', name='Peptides-func', split='train')
d = tr[0]
print('train graphs:', len(tr), '| example:', d)
sizes = [tr[i].num_nodes for i in range(2000)]
print(f'nodes: median {int(np.median(sizes))}, max {max(sizes)} | labels: {d.y.shape[-1]} classes')

## 2. Do these molecules have parallel paths? (Yes.)

Rings and branches in peptides create parallel walks — pairs `(i,j)` with `(A^g)[i,j] ≥ 2`, which the kQ/I
ideal would collapse. So the tie below is *not* from a lack of parallel structure; it is because the
over-squashing is mild on these small graphs.

In [ ]:
import scipy.sparse as sp
par = {g: 0 for g in range(2, 5)}
for idx in range(500):
    e = tr[idx].edge_index.numpy(); n = tr[idx].num_nodes
    A = sp.csr_matrix((np.ones(e.shape[1]), (e[0], e[1])), shape=(n, n)).toarray()
    P = A @ A
    for g in range(2, 5):
        par[g] += int((P >= 2).sum()); P = P @ A
for g in range(2, 5):
    print(f'range g={g}: {par[g]/500:.0f} parallel-path pairs per molecule on average')

## 3. Fast walk-reachability supports (sparse matrix powers)

The exact path-algebra enumeration is too slow for 15k molecules (~53 s/graph). Walk Attention only needs the
*support* (which pairs are walk-reachable) = the nonzero pattern of `A^g`, computed by sparse matrix powers in
~0.2 ms/graph (whole dataset in ~11 s). The support stays sparse on peptides (1–8% at range 3).

In [ ]:
from oversquash.lrgb import AttachLRGBMasks
tf = AttachLRGBMasks(n_layers=3)
dm = tf(tr[0])
for g, m in enumerate(dm.walk_masks):
    nz = m._nnz(); N = dm.num_nodes
    print(f'range {g+1}: support {nz}/{N*N} = {100*nz/(N*N):.0f}% dense')

## 4. Train and evaluate (GCN / GAT / WalkAttention)

Full training (40 epochs on ~11k graphs) is in `run_lrgb.py` (run from the repo root). The cell below runs a
**short demo** on a subset so the notebook executes quickly; for the reported numbers use `run_lrgb.py`.

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader
from oversquash.lrgb import collate_lrgb, LRGBNet, average_precision

tf = AttachLRGBMasks(n_layers=3)
sub = [tf(tr[i]) for i in range(1500)]          # subset for a fast demo
trl = DataLoader(sub[:1200], batch_size=64, shuffle=True, collate_fn=collate_lrgb)
val = DataLoader(sub[1200:], batch_size=64, collate_fn=collate_lrgb)

for bb in ['gcn', 'gat', 'walkattn']:
    torch.manual_seed(0)
    m = LRGBNet(bb, hidden=64, out_dim=10, n_layers=3, n_heads=4, dropout=0.1)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-4)
    for e in range(15):
        m.train()
        for b in trl:
            opt.zero_grad(); F.binary_cross_entropy_with_logits(m(b), b.y).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    print(f'{bb:9s} demo val AP (1.5k subset, 15 ep) = {average_precision(m, val):.3f}')

## 5. Full result (from `run_lrgb_multiseed.py`, 4 seeds — `results/tables/lrgb_peptides_func_4seeds.csv`)

| Model | test AP (mean ± std, 4 seeds) |
|---|---|
| GCN | 0.492 ± 0.011 |
| GAT | 0.526 ± 0.004 |
| **Walk Attention** | **0.526 ± 0.007** |

Walk Attention beats GCN and is close to GAT — **comparable, not superior** on this mild-over-squashing benchmark.
The parallel paths are there (Step 2) and the kQ/I ideal captures them, but mild squashing means multi-hop
reach buys little. For the regime where Walk Attention wins decisively (clear parallel paths + *severe*
squashing), see **RingTransfer** (`run_ring.py`, notebook 06).